# Bronze: register and verify LF v2 sources

Run top to bottom in a new authenticated **lakehouse_to_ai** AIDP Spark session.
Attach the **aidp_lakehouse_to_ai-0.4.0-py3-none-any.whl** wheel to compute and restart
before running. Requires Standard catalog/schema privileges and read access to
the existing OCI initial entity/control objects. No credentials belong here.

Expected: customers **600**, orders **30,600**, deliveries **30,000**, events
**94,572**, zero corrupt records and zero full-row differences in both directions.
Existing compatible registrations are preserved. A v1 registration stops this flow:
use the separate reviewed migration; this notebook never drops/replaces tables.

Notebook 00 is an optional catalog-only prerequisite/check for administrators;
this notebook includes the same idempotent setup, so running 00 is unnecessary.
See `docs/bronze-v2-migration.md` for rollback and `docs/aidp-bronze-setup.md` for access.

In [ ]:
# CONFIGURATION: edit environment values only here.
# These are non-secret values verified in docs/object-storage-execution.md.
CONFIG = {
    "catalog": "lakehouse_demo",
    "namespace": "frgpslcqn7sm",  # Object Storage namespace, NOT oraemeaaidata2
    "bucket": "CristinaVaras-bucket-AIDP",
    "prefix": "demos/lakehouse-to-ai/source-data/v2/seed-42/reference-2026-10-28",
    "reference_date": "2026-10-28",  # Fixed synthetic-data reference date
    "seed": 42,
    "batch_id": "initial",         # Do not include incremental in this milestone
}

## 1. Runtime and configuration

In [ ]:
# Fail clearly even if the wheel or attached Spark session is missing.
try:
    from aidp_lakehouse.bronze_workflow import BronzeRun
    run = BronzeRun(spark, CONFIG)
except Exception as error:
    print(f"BRONZE FAIL [initialization]: {type(error).__name__}: {error}")
    raise

## 2. Create or verify the Standard catalog and schemas

In [ ]:
run.setup()

## 3. Read and validate control documents

In [ ]:
run.controls()

## 4. Register or preserve the four external tables

In [ ]:
run.register()

## 5. Full direct-reader comparison

In [ ]:
run.verify()

## 6. Small raw-data and provenance samples

In [ ]:
run.samples()

## 7. Final result

In [ ]:
verification = run.finish()

Continue to **02_build_silver.ipynb** only after BRONZE PASS. Use its matching v2 configuration;
Silver business logic is unchanged. A failure prints BRONZE FAIL with the failing stage and reason,
raises an exception and stops execution. Fix the cause and restart from the top.
Do not skip a failed cell or use enforceSchema=true, trimming or monkeypatches.